<a href="https://colab.research.google.com/github/ddickson28/Captain-FPSO-BN/blob/main/TrialBN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mbnpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.3/105.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 8.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you h

In [1]:
#Import modules
import numpy as np #importing a module
from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy
import itertools
from scipy.stats import truncnorm
from math import erf, sqrt

In [23]:
"Building the BN and relationship"
from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy

#1 Define the variables (nodes). N1, N2, ... ,Nn. Child node last


WeatherDamage = variable.Variable('WeatherDamage', ['0 to 0.2','0.2 to 0.4','0.4 to 0.6','0.6 to 0.8','0.8 to 1.0']) #Damage taken from BV report, discretised over 5 intervals
LoadingDamage = variable.Variable('LoadingDamage', ['0 to 0.2','0.2 to 0.4','0.4 to 0.6','0.6 to 0.8','0.8 to 1.0']) #Damage taken from BV report, discretised over 5 intervals
TotalDamage = variable.Variable('TotalDamage', ['0 to 0.2','0.2 to 0.4','0.4 to 0.6','0.6 to 0.8','0.8 to 1.0']) #Mean is sum from weather and loading damage above, discretised over 5 intervals
PreviousRepair = variable.Variable('PreviousRepair', ['False','True']) #Repaired or not

RemainingDamage = variable.Variable('RemainingDamage', ['No 0 - 0.2','No 0.2 - 0.4','No 0.4 - 0.6','No 0.6 - 0.8','No 0.8 - 1.0',
																												'Yes 0 - 0.2','Yes 0.2 - 0.4','Yes 0.4 - 0.6','Yes 0.6 - 0.8','Yes 0.8 - 1.0']) # 1 - TotalDamage or// Previous step damage - TotalDamage


CrackLocationInterest = variable.Variable('CrackLocationInterest', ['No, 0.2 Yes', 'No, 0.2,No',
                                                                    'No, 0.4 Yes', 'No, 0.4, No',
																																		'No, 0.6 Yes', 'No, 0.6, No',
																																		'No, 0.8 Yes', 'No, 0.8, No',
		 																																'No, 1.0 Yes', 'No, 1.0, No',
                                                                    'Yes, 0.2 Yes', 'Yes, 0.2,No',
																																		'Yes, 0.4 Yes', 'No, 0.4, No',
																																		'Yes, 0.6 Yes', 'No, 0.6, No',
		 																																'Yes, 0.8 Yes', 'No, 0.8, No',
		 																																'Yes, 1.0 Yes', 'No, 1.0, No' ])


#2 Define the cpm of the variables from 1 in order specified above.


cpm_WeatherDamage = cpm.Cpm(
									[WeatherDamage], no_child=1,
									 C=np.array([[0],[1],[2],[3],[4]], dtype=int),
									 #p=np.array([0.017,0.435,0.518,0.03,0.0]) #This needs to be from Excel Norm Distribution
)

cpm_LoadingDamage = cpm.Cpm(
									[LoadingDamage], no_child=1,
									 C=np.array([[0],[1],[2],[3],[4]], dtype=int),
									 p=np.array([0.122,0.677,0.198,0.002,0.0]) #This needs to be from excel Norm Distribution
)


cpm_TotalDamage = cpm.Cpm(
									[TotalDamage], no_child=1,
									 C=np.array([[0],[1],[2],[3],[4]], dtype=int),
									 p=np.array([0.0000001,0.0005,0.0999,0.6641,0.2322])
)


cpm_PreviousRepair = cpm.Cpm(
									[PreviousRepair], no_child=1,
									 C=np.array([[0],[1]], dtype=int),
									 p=np.array([0.5, 0.5]) #Assume that there hasn't been a repair unless specified otherwise
)

cpm_RemainingDamage = cpm.Cpm(
									[RemainingDamage], no_child=1,
									C=np.array([[0],[1],[2],[3],[4],[5],[6],[7],[8],[9]], dtype=int),
									p=np.array([0.5673, 0.3788, 0.0113, 0.0000, 0.0000, 0.2322, 0.6641, 0.0999, 0.0005, 0.0005 ])
)

cpm_CrackLocationInterest = cpm.Cpm(
									[CrackLocationInterest, RemainingDamage, PreviousRepair, TotalDamage, LoadingDamage, WeatherDamage], no_child=1,
									 C=np.array([[0],[1],[2],[3],[4],[5],[6],[7],[8],[9],[10],[11],[12],[13],[14],[15],[16],[17],[18],[19]], dtype=int),
									 p=np.array([0.818, 0.812, 0.175, 0.825, 0.008, 0.992, 0, 1, 0, 1, 0.818, 0.182, 0.175, 0.825 ,0.008, 0.992, 0, 1 ,0 ,1])
)

print(cpm_CrackLocationInterest)

AssertionError: C must have the same number of columns as that of variables